In [438]:
!pip install -qq pandas numpy sklearn xgboost imblearn tabulate

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [439]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, OrdinalEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from tabulate import tabulate


pd.set_option('display.max_columns', 200)


ModuleNotFoundError: No module named 'tabulate'

# Load

In [393]:
app_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/application_record.csv")
cred_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/credit_record.csv")

In [394]:
app_df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438552,6840104,M,N,Y,0,135000.0,Pensioner,Secondary / secondary special,Separated,House / apartment,-22717,365243,1,0,0,0,NaN,1.0
438553,6840222,F,N,N,0,103500.0,Working,Secondary / secondary special,Single / not married,House / apartment,-15939,-3007,1,0,0,0,Laborers,1.0
438554,6841878,F,N,N,0,54000.0,Commercial associate,Higher education,Single / not married,With parents,-8169,-372,1,1,0,0,Sales staff,1.0
438555,6842765,F,N,Y,0,72000.0,Pensioner,Secondary / secondary special,Married,House / apartment,-21673,365243,1,0,0,0,NaN,2.0


In [395]:
cred_df

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C
...,...,...,...
1048570,5150487,-25,C
1048571,5150487,-26,C
1048572,5150487,-27,C
1048573,5150487,-28,C


In [396]:
cred_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   ID              1048575 non-null  int64 
 1   MONTHS_BALANCE  1048575 non-null  int64 
 2   STATUS          1048575 non-null  object
dtypes: int64(2), object(1)
memory usage: 24.0+ MB


# Creating Target

In [397]:
is_bad = (
    cred_df['STATUS']
    .isin(['1', '2', '3','4','5'])
    .groupby(cred_df['ID'])
    .max()  # Jika ada minimal 1 True, return 1
    .astype(int)
)

# Aggregrating Credit Records

In [398]:
credit_agg = cred_df.groupby('ID').agg(
    CREDIT_HISTORY_LENGTH=('MONTHS_BALANCE', lambda x: abs(x.min() - x.max()) + 1),  # Panjang riwayat kredit
    BAD_DEBT_RATIO=('STATUS', lambda x: (x.isin(['1', '2', '3', '4', '5']).sum()) / len(x)),  # Proporsi keterlambatan pembayaran tagihan >30 hari
    AVERAGE_DELAYED_MONTHS=('STATUS', lambda x: x[x.isin(['1', '2', '3', '4', '5'])].astype(int).mean() if x.isin(['1', '2', '3', '4', '5']).any() else 0),
).reset_index()
    
# Gabungkan target dengan fitur agregasi kredit
credit_agg['IS_BAD'] = is_bad.values

In [399]:
credit_agg

,ID,CREDIT_HISTORY_LENGTH,BAD_DEBT_RATIO,AVERAGE_DELAYED_MONTHS,IS_BAD
0,5001711,4,0.0,0.0,0
1,5001712,19,0.0,0.0,0
2,5001713,22,0.0,0.0,0
3,5001714,15,0.0,0.0,0
4,5001715,60,0.0,0.0,0
...,...,...,...,...,...
45980,5150482,18,0.0,0.0,0
45981,5150483,18,0.0,0.0,0
45982,5150484,13,0.0,0.0,0
45983,5150485,2,0.0,0.0,0


# Merge Data

In [400]:
new_df = pd.merge(app_df, credit_agg, on="ID", how="inner")

In [401]:
new_df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,CREDIT_HISTORY_LENGTH,BAD_DEBT_RATIO,AVERAGE_DELAYED_MONTHS,IS_BAD
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,16,0.062500,1.000000,1
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,15,0.066667,1.000000,1
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,30,0.000000,0.000000,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,-17348,-2420,1,0,0,0,Managers,2.0,12,0.333333,4.750000,1
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,24,0.750000,2.944444,1
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,33,0.545455,2.944444,1
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,-17958,-655,1,0,0,0,Sales staff,2.0,10,0.200000,1.500000,1


In [402]:
new_df.isna().sum()

ID                            0
CODE_GENDER                   0
FLAG_OWN_CAR                  0
FLAG_OWN_REALTY               0
CNT_CHILDREN                  0
AMT_INCOME_TOTAL              0
NAME_INCOME_TYPE              0
NAME_EDUCATION_TYPE           0
NAME_FAMILY_STATUS            0
NAME_HOUSING_TYPE             0
DAYS_BIRTH                    0
DAYS_EMPLOYED                 0
FLAG_MOBIL                    0
FLAG_WORK_PHONE               0
FLAG_PHONE                    0
FLAG_EMAIL                    0
OCCUPATION_TYPE           11323
CNT_FAM_MEMBERS               0
CREDIT_HISTORY_LENGTH         0
BAD_DEBT_RATIO                0
AVERAGE_DELAYED_MONTHS        0
IS_BAD                        0
dtype: int64

In [403]:
new_df = new_df.dropna()

In [404]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25134 entries, 2 to 36456
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      25134 non-null  int64  
 1   CODE_GENDER             25134 non-null  object 
 2   FLAG_OWN_CAR            25134 non-null  object 
 3   FLAG_OWN_REALTY         25134 non-null  object 
 4   CNT_CHILDREN            25134 non-null  int64  
 5   AMT_INCOME_TOTAL        25134 non-null  float64
 6   NAME_INCOME_TYPE        25134 non-null  object 
 7   NAME_EDUCATION_TYPE     25134 non-null  object 
 8   NAME_FAMILY_STATUS      25134 non-null  object 
 9   NAME_HOUSING_TYPE       25134 non-null  object 
 10  DAYS_BIRTH              25134 non-null  int64  
 11  DAYS_EMPLOYED           25134 non-null  int64  
 12  FLAG_MOBIL              25134 non-null  int64  
 13  FLAG_WORK_PHONE         25134 non-null  int64  
 14  FLAG_PHONE              25134 non-null  int

In [405]:
new_df["IS_BAD"].value_counts(normalize=True)

IS_BAD
0    0.877099
1    0.122901
Name: proportion, dtype: float64

In [406]:
new_df["IS_BAD"].value_counts()

IS_BAD
0    22045
1     3089
Name: count, dtype: int64

In [407]:
for column in new_df.select_dtypes(include='object'):
    unique_values = new_df[column].unique()
    print(f"Unique values in {column}: {unique_values}")

Unique values in CODE_GENDER: ['M' 'F']
Unique values in FLAG_OWN_CAR: ['Y' 'N']
Unique values in FLAG_OWN_REALTY: ['Y' 'N']
Unique values in NAME_INCOME_TYPE: ['Working' 'Commercial associate' 'State servant' 'Student' 'Pensioner']
Unique values in NAME_EDUCATION_TYPE: ['Secondary / secondary special' 'Higher education' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Unique values in NAME_FAMILY_STATUS: ['Married' 'Single / not married' 'Civil marriage' 'Separated' 'Widow']
Unique values in NAME_HOUSING_TYPE: ['House / apartment' 'Rented apartment' 'Municipal apartment'
 'With parents' 'Co-op apartment' 'Office apartment']
Unique values in OCCUPATION_TYPE: ['Security staff' 'Sales staff' 'Accountants' 'Laborers' 'Managers'
 'Drivers' 'Core staff' 'High skill tech staff' 'Cleaning staff'
 'Private service staff' 'Cooking staff' 'Low-skill Laborers'
 'Medicine staff' 'Secretaries' 'Waiters/barmen staff' 'HR staff'
 'Realty agents' 'IT staff']


# Renaming Feature

In [408]:
new_df = new_df.rename(columns={
    "DAYS_BIRTH" : "AGE",
    "NAME_FAMILY_STATUS" : "MARITAL_STATUS",
    "NAME_HOUSING_TYPE" :  "DWELLING_TYPE",
    "AMT_INCOME_TOTAL" : "ANNUAL_INCOME",
    "OCCUPATION_TYPE" : "JOB_TITLE",
    "NAME_EDUCATION_TYPE" : "EDUCATION_LEVEL",
    "DAYS_EMPLOYED" : "EMPLOYMENT_LENGTH",
    "NAME_INCOME_TYPE" : "EMPLOYMENT_STATUS"})

In [409]:
new_df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,ANNUAL_INCOME,EMPLOYMENT_STATUS,EDUCATION_LEVEL,MARITAL_STATUS,DWELLING_TYPE,AGE,EMPLOYMENT_LENGTH,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,JOB_TITLE,CNT_FAM_MEMBERS,CREDIT_HISTORY_LENGTH,BAD_DEBT_RATIO,AVERAGE_DELAYED_MONTHS,IS_BAD
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,30,0.000000,0.000000,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,5,0.000000,0.000000,0
5,5008810,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,27,0.000000,0.000000,0
6,5008811,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,39,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,-17348,-2420,1,0,0,0,Managers,2.0,12,0.333333,4.750000,1
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,24,0.750000,2.944444,1
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,33,0.545455,2.944444,1
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,-17958,-655,1,0,0,0,Sales staff,2.0,10,0.200000,1.500000,1


# Drop ID 

In [410]:
new_df = new_df.drop(["ID"], axis=1)

# Train Test Split

In [411]:
train_original, test_original = train_test_split(new_df, test_size=0.2, stratify=new_df["IS_BAD"], random_state=69)

In [412]:
train_original["IS_BAD"].value_counts()

IS_BAD
0    17636
1     2471
Name: count, dtype: int64

In [413]:
test_original["IS_BAD"].value_counts()

IS_BAD
0    4409
1     618
Name: count, dtype: int64

In [414]:
# Save Train and Test Original
train_original.to_csv("D:\File Gaung\Kuliah TIF UB\BCC\Intern 2025\Project\Dataset\processed\\train_original_dataset.csv", index=False)
test_original.to_csv("D:\File Gaung\Kuliah TIF UB\BCC\Intern 2025\Project\Dataset\processed\\test_original_dataset.csv", index=False)

<>:2: SyntaxWarning: invalid escape sequence '\F'
<>:3: SyntaxWarning: invalid escape sequence '\F'
<>:2: SyntaxWarning: invalid escape sequence '\F'
<>:3: SyntaxWarning: invalid escape sequence '\F'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14892\1431264067.py:2: SyntaxWarning: invalid escape sequence '\F'
  train_original.to_csv("D:\File Gaung\Kuliah TIF UB\BCC\Intern 2025\Project\Dataset\processed\\train_original_dataset.csv", index=False)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14892\1431264067.py:3: SyntaxWarning: invalid escape sequence '\F'
  test_original.to_csv("D:\File Gaung\Kuliah TIF UB\BCC\Intern 2025\Project\Dataset\processed\\test_original_dataset.csv", index=False)


In [415]:
# buat copy untuk membuat pipeline, supaya yang original tetap tak tersentuh
train_copy = train_original.copy()
test_copy = test_original.copy()

In [416]:
train_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20107 entries, 16886 to 22930
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CODE_GENDER             20107 non-null  object 
 1   FLAG_OWN_CAR            20107 non-null  object 
 2   FLAG_OWN_REALTY         20107 non-null  object 
 3   CNT_CHILDREN            20107 non-null  int64  
 4   ANNUAL_INCOME           20107 non-null  float64
 5   EMPLOYMENT_STATUS       20107 non-null  object 
 6   EDUCATION_LEVEL         20107 non-null  object 
 7   MARITAL_STATUS          20107 non-null  object 
 8   DWELLING_TYPE           20107 non-null  object 
 9   AGE                     20107 non-null  int64  
 10  EMPLOYMENT_LENGTH       20107 non-null  int64  
 11  FLAG_MOBIL              20107 non-null  int64  
 12  FLAG_WORK_PHONE         20107 non-null  int64  
 13  FLAG_PHONE              20107 non-null  int64  
 14  FLAG_EMAIL              20107 non-null 

# Transform Data 

ID:
* Drop the feature

CODE_GENDER: 
* One hot encoding

AGE from DAYS_BIRTH: 
* Min-max scaling
* Fix skewness
* Abs value and div 365.25

MARITAL_STATUS from NAME_FAMILY_STATUS: 
* One hot encoding

CNT_FAM_MEMBERS:
* Fix outliers

CNT_CHILDERN:
* Fix outliers
* Drop feature

DWELLING_TYPE from NAME_HOUSING_TYPE 
* One hot encoding

ANNUAL_INCOME from AMT_INCOME_TOTAL:
* Remove outliers
* Fix skewness
* Min-max scaling

JOB_TITLE from OCCUPATION_TYPE 
* One hot encoding


EMPLOYMENT_STATUS from NAME_INCOME_TYPE: 
* One hot encoding

EDUCATION_LEVEL from NAME_EDUCATION TYPE: 
* Ordinal encoding

EMPLOYMENT_LENGTH from DAYS_EMPLOYED: 
* Remove outliers
* Min-max scaling
* Abs value and div 365.25
* change days of employments of retirees to 0

FLAG_OWN_CAR: 
* Change it numerical
* One-hot encoding

FLAG_OWN_REALTY: 
* Change it numerical
* One-hot encoding

FLAG_MOBILE: 
* Drop feature (because all of it is 1) 

FLAG_WORK_PHONE : 
* One-hot encoding

FLAG_PHONE: 
* One-hot encoding

FLAG_EMAIL: 
* One-hot encoding

IS_BAD: 
* balance the data with SMOTE

## Feature Engineering

In [417]:
# Fungsi untuk AGE
def transform_age(X):
    return abs(X) / 365.25

In [418]:
def transform_employment_length(X):
    return np.where(X > 0, 0, abs(X) / 365.25)

## Remove Outliers  

In [419]:
def remove_outliers_transform(X):
    Q1 = np.quantile(X, 0.25, axis=0)
    Q3 = np.quantile(X, 0.75, axis=0)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return np.clip(X, lower_bound, upper_bound)

## One-Hot Encoding

In [420]:
categorical_features = [
    'CODE_GENDER', 'MARITAL_STATUS', 'DWELLING_TYPE', 
    'JOB_TITLE', 'EMPLOYMENT_STATUS', 'FLAG_OWN_CAR', 
    'FLAG_OWN_REALTY', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL'
]

one_hot_encoder = OneHotEncoder(drop='first', sparse_output=False)


## Ordinal Encoding

In [421]:
education_order = ['Lower secondary', 'Secondary / secondary special', 'Incomplete higher', 'Higher education', 'Academic degree']
ordinal_encoder = OrdinalEncoder(categories=[education_order])

## Features Scalling

In [422]:
numerical_features = ['AGE', 'EMPLOYMENT_LENGTH', 'ANNUAL_INCOME']
scaler = MinMaxScaler()

In [423]:
train_copy.shape

(20107, 21)

# Pipeline

In [424]:
preprocessor = ColumnTransformer(
    transformers=[
        ('age_transform', FunctionTransformer(transform_age), ['AGE']),
        ('employment_length_transform', FunctionTransformer(transform_employment_length), ['EMPLOYMENT_LENGTH']),
        ('income_outliers', FunctionTransformer(remove_outliers_transform), ['ANNUAL_INCOME']),
        ('family_members_outliers', FunctionTransformer(remove_outliers_transform), ['CNT_FAM_MEMBERS']),
        ('num', scaler, numerical_features),
        ('cat', one_hot_encoder, categorical_features),
        ('edu', ordinal_encoder, ['EDUCATION_LEVEL'])
    ], remainder="drop" # menghapus kolom lain yang tidak di proses
)

In [425]:
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

In [426]:
train_copy_prep = pipeline.fit_transform(train_copy)

In [427]:
train_copy_prep

array([[5.73634497e+01, 3.16221766e+00, 1.22400000e+05, ...,
        0.00000000e+00, 1.00000000e+00, 1.00000000e+00],
       [5.40177960e+01, 1.35085558e+01, 1.35000000e+05, ...,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00],
       [4.57796030e+01, 2.42984257e+01, 9.90000000e+04, ...,
        1.00000000e+00, 0.00000000e+00, 3.00000000e+00],
       ...,
       [4.08898015e+01, 4.79945243e+00, 1.35000000e+05, ...,
        1.00000000e+00, 0.00000000e+00, 1.00000000e+00],
       [3.27173169e+01, 1.10116359e+01, 1.80000000e+05, ...,
        1.00000000e+00, 0.00000000e+00, 1.00000000e+00],
       [2.83723477e+01, 3.50992471e+00, 3.60000000e+05, ...,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00]],
      shape=(20107, 44))

# Baseline Model

In [428]:
X = train_copy.drop(columns=['IS_BAD'])  # Fitur
y = train_copy['IS_BAD']  # Target

In [429]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [432]:

classifiers = {
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "SVM": SVC(kernel='rbf', C=1, random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=100),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
}

In [433]:
# Define business value scores
TP_value = 200
TN_value = 0
FP_value = -50
FN_value = 100


def train_result(X_train, X_test, y_train, y_test):
    results = []
    trained_models = {}

    for name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)

        # Calculate TP, TN, FP, and FN
        tn, fp, fn, tp = cm.ravel()

        # Calculate precision and recall
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1_score = fbeta_score(y_test, y_pred, beta=1)
        f2_score = fbeta_score(y_test, y_pred, beta=2)

        # Calculate business value score
        business_value_score = (tp * TP_value) + (tn * TN_value) + (fp * FP_value) + (fn * FN_value)

        # Format results and append to the list
        results.append([name, precision, recall, f1_score, f2_score, tp, tn, fp, fn, business_value_score])
        
        # Save trained models
        trained_models[name] = clf

    # Create a table using tabulate
    headers = ["Classifier", "Precision", "Recall", "F1 Score", "F2 Score", "TP", "TN", "FP", "FN", "Business Value Score"]
    table = tabulate(results, headers=headers)

    # Print the table
    print(table)
    
    # Return trained models
    return trained_models

In [434]:
train_result(X_train, X_test, y_train, y_test)

ValueError: could not convert string to float: 'F'

## SMOTE Oversampling

In [ ]:
X = train_copy.drop(columns=['IS_BAD'])  # Fitur
y = train_copy['IS_BAD']  # Target

In [ ]:
from imblearn.over_sampling import SMOTE

# Jalankan preprocessing
X_transformed = pipeline.fit_transform(X)

# Terapkan SMOTE untuk menangani ketidakseimbangan kelas
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)


In [ ]:
X_resampled

array([[0.78356796, 0.07274354, 0.28648649, ..., 0.        , 1.        ,
        1.        ],
       [0.71128593, 0.3143058 , 0.32432432, ..., 0.        , 0.        ,
        1.        ],
       [0.53330179, 0.56622347, 0.21621622, ..., 1.        , 0.        ,
        3.        ],
       ...,
       [0.52170827, 0.05382255, 0.17567568, ..., 0.        , 0.        ,
        1.        ],
       [0.28777635, 0.18644669, 0.57217189, ..., 0.        , 0.        ,
        1.        ],
       [0.68935917, 0.2379284 , 0.70156678, ..., 0.        , 0.        ,
        1.        ]], shape=(35272, 40))

In [ ]:
y_resampled.value_counts()

IS_BAD
0    17636
1    17636
Name: count, dtype: int64